# Конспект. Модуль 7: LightGBM изнутри

## 1. Зачем это нужно и как это связано с предыдущими модулями

Это **ключевой модуль всего курса** — именно LightGBM явно указан как обязательное требование в вашей вакансии, и именно его вы будете использовать в проекте FraudGuard. В Модуле 6 мы разобрали общий математический фундамент, на котором строится LightGBM: регуляризованная целевая функция, использование градиента **и** гессиана (метод Ньютона), гистограммный поиск сплитов. LightGBM (Microsoft, 2017) берёт этот фундамент и добавляет **три собственных инженерных нововведения**, цель которых — сделать бустинг быстрым и экономным по памяти на **очень больших** данных (миллионы строк, сотни признаков — ровно ваш случай с IEEE-CIS):

1. **Leaf-wise рост дерева** вместо Level-wise.
2. **GOSS** — умное сэмплирование объектов по величине градиента.
3. **EFB** — объединение разреженных взаимоисключающих признаков.

Разберём каждое досконально, с числовыми примерами.

## 2. Гистограммный алгоритм: развёрнутая версия

В Модуле 6 мы уже познакомились с идеей гистограмм на общем уровне (биннинг признаков вместо точного перебора). Здесь — дополнительная деталь, специфичная для LightGBM/современных реализаций, которую стоит знать для собеседования: **трюк вычитания гистограмм (histogram subtraction trick)**.

### 2.1. Идея

Гистограмма узла — это, по сути, массив накопленных сумм `G` и `H` (сумм градиентов и гессианов, Модуль 6) по каждой корзине признака. Построение гистограммы для узла требует одного прохода `O(N_узла)` по всем объектам, попавшим в этот узел.

Когда узел разбивается на левого и правого потомка, справедливо простое алгебраическое тождество (гистограмма — это просто сумма по подмножествам, а множества левого и правого потомка **в сумме составляют** множество родителя, не пересекаясь):

In [ ]:
Гистограмма(родитель) = Гистограмма(левый потомок) + Гистограмма(правый потомок)

**Следствие:** если мы уже построили гистограмму родителя (она была нужна на предыдущем шаге, чтобы решить, каким узлом он вообще стал) и построили гистограмму **одного** из потомков (обычно выбирают **меньший** по числу объектов, чтобы построение было максимально дешёвым), то гистограмма **второго** потомка получается **бесплатно**, простым вычитанием — без единого дополнительного прохода по его данным:

In [ ]:
Гистограмма(больший потомок) = Гистограмма(родитель) - Гистограмма(меньший потомок)

**Численная иллюстрация.** Пусть у признака 3 корзины, и гистограмма родителя (суммы градиентов по корзинам) — `[10, 15, 20]` (всего 3 корзины). Разбиение отправило часть объектов в левый потомок; посчитав по нему (проход по его, меньшей, части данных), получили гистограмму левого — `[4, 6, 9]`. Гистограмма правого потомка получается мгновенно:

In [ ]:
[10-4, 15-6, 20-9] = [6, 9, 11]

**Экономия:** вместо двух проходов `O(N_левый) + O(N_правый) = O(N_родитель)` нужен только один проход по **меньшему** потомку — фактически, эта оптимизация примерно **вдвое** снижает объём вычислений при построении гистограмм на каждом уровне дерева, что для сотен деревьев на миллионах строк даёт заметный суммарный выигрыш по времени.

## 3. Leaf-wise рост дерева vs Level-wise

### 3.1. Определения

**Level-wise (по уровням)** — стратегия классического CART и XGBoost по умолчанию: дерево растёт **уровень за уровнем** — сначала разбиваются **все** узлы текущей глубины (даже если некоторые из этих разбиений дают маленький выигрыш), только потом дерево переходит на следующую глубину.

**Leaf-wise (best-first, по листу с максимальным Gain)** — стратегия LightGBM: на каждом шаге ищется **тот единственный лист во всём текущем дереве** (независимо от его глубины), разбиение которого даёт **максимальный** Gain (формула из Модуля 6), и разбивается **только он**. Дерево растёт «жадно вширь по качеству», а не «равномерно по глубине».

### 3.2. Численный пример, показывающий разницу

Представим дерево на промежуточном этапе роста: уже сделан один сплит (корень разбит на узлы `A` и `B`). Для каждого из них известен потенциальный Gain от **дальнейшего** разбиения (посчитанный по формуле Модуля 6 на основе накопленных гистограмм):

In [ ]:
Gain(разбить A дальше) = 50
Gain(разбить B дальше) = 5

`A` содержит существенно больше «полезной структуры», которую ещё стоит выделить; `B` уже почти однороден, и дальнейшее разбиение почти не помогает.

**Сценарий с ограничением на общее число листьев `num_leaves=3`** (то есть после начального сплита `{A,B}` — уже 2 листа — бюджет позволяет ровно **ещё один** сплит):

- **Leaf-wise:** сравнивает `Gain(A)=50` и `Gain(B)=5`, жадно выбирает **лучший** — разбивает `A` на `A1, A2`. Итоговые листья: `{A1, A2, B}` — 3 листа, использован самый ценный из двух доступных сплитов, суммарный захваченный выигрыш `= 50`.
- **Level-wise:** устроен принципиально иначе — он не умеет «выбирать» один сплит из двух на одном уровне, он либо разбивает **весь** текущий уровень целиком, либо не разбивает **вообще**. При `max_depth=2` он обязан разбить **и** `A`, **и** `B` — но это даёт **4** листа `{A1,A2,B1,B2}`, что превышает бюджет в 3 листа. Если же ограничиться `max_depth=1` (не заходить на второй уровень вообще), получаем только `{A,B}` — 2 листа, **вообще не захватив** ценный `Gain=50` от разбиения `A`.

**Вывод, который стоит проговорить дословно на собеседовании:** *«При одинаковом бюджете листьев leaf-wise почти всегда достигает более низкого loss, чем level-wise, потому что жадно расходует бюджет на разбиения с наибольшим Gain, а не тратит его равномерно на весь уровень независимо от качества конкретных разбиений»*. Level-wise можно рассматривать как более «грубый» инструмент, который выигрывает свою полезность только при полном использовании всего уровня — если полезны **оба** потенциальных сплита одного уровня, level-wise и leaf-wise в итоге дадут одинаковый результат (в этом легко убедиться: при бюджете `num_leaves=4` в нашем примере оба подхода в итоге разобьют и `A`, и `B`, получив одинаковые 4 листа) — разница проявляется именно там, где **бюджет ограничен и не позволяет забрать все доступные сплиты**.

### 3.3. Побочный эффект: асимметрия дерева и риск переобучения

Из-за жадного выбора «лучшего листа», а не «лучшего уровня», деревья LightGBM обычно получаются **несимметричными**: одна ветвь может уйти очень глубоко (продолжая находить выгодные разбиения), а другая — остаться почти нетронутой. Это прямое следствие того, что `num_leaves` контролирует **число листьев**, а не глубину — при неограниченной глубине (`max_depth=-1`, что и есть значение по умолчанию в LightGBM) leaf-wise может **сколь угодно глубоко** углубляться в одну конкретную, узкую, но статистически выгодную на данный момент подвыборку данных — а это уже прямой путь к переобучению: разбиение, «выгодное» на крошечной подгруппе объектов, часто отражает шум этой конкретной подгруппы, а не настоящую закономерность (тот же принцип, что мы разбирали для одиночного глубокого дерева в Модуле 1).

**Именно поэтому `num_leaves` — главный, а не вспомогательный регулятор сложности в LightGBM** (в отличие от XGBoost/классического CART, где основной контроль — через `max_depth`): при leaf-wise росте глубина сама по себе — плохой индикатор сложности (дерево может быть глубоким лишь в одной узкой ветке, оставаясь в целом простым), а вот **число листьев** прямо и однозначно определяет, сколько «независимых решающих правил» в итоге содержит дерево.

## 4. GOSS: Gradient-based One-Side Sampling

### 4.1. Мотивация

Даже с гистограммным алгоритмом (раздел 2) построение гистограммы узла требует **прохода по всем объектам**, попавшим в этот узел — `O(N)`. На датасетах с миллионами строк (ваш IEEE-CIS: ~590K, а в проде — намного больше) это остаётся узким местом, повторяющимся на каждом узле каждого дерева. `subsample` из Модуля 5 уже даёт частичное решение (случайно берём подмножество строк), но это **грубый** инструмент — он одинаково безжалостно выбрасывает как «неважные», так и потенциально «очень важные» строки.

**Ключевая идея GOSS:** не все объекты одинаково полезны для построения дерева **на текущей итерации**. Объекты с **большим по модулю градиентом** — те, где текущая модель **сильно ошибается** — несут больше информации о том, куда двигаться дальше (вспомните Модуль 3: градиент — это буквально «насколько и в какую сторону нужно исправить предсказание»). Объекты с **маленьким градиентом** модель уже предсказывает хорошо — их вклад в решение, куда двигаться дальше, минимален.

### 4.2. Алгоритм

1. Посчитать `|g_i|` (модуль градиента) для всех `N` объектов на текущей итерации.
2. Отсортировать по убыванию `|g_i|` и взять верхние `a·100%` объектов (`top_rate`, например `a=0.2`) — это множество `A` («трудные» объекты), **сохраняются все, без исключения**.
3. Из оставшихся `(1-a)·N` объектов **случайно** выбрать долю `b` от **общего** `N` (`other_rate`, например `b=0.1`) — множество `B`.
4. Чтобы итоговая сумма градиентов по множеству `B` (используемая при построении гистограмм) оставалась **несмещённой оценкой** истинной суммы по всем `(1-a)·N` «лёгким» объектам, каждый отобранный объект из `B` **домножается на компенсирующий коэффициент**:

In [ ]:
множитель = (1 - a) / b

5. Строить гистограммы (суммы `G`, `H` по корзинам) **только** по объединению `A ∪ B` (с указанным множителем для `B`), а не по всем `N` объектам.

### 4.3. Почему множитель именно такой — вывод

Пусть в «лёгкой» группе всего `(1-a)N` объектов, а мы случайно выбираем из неё `bN` (то есть долю `b/(1-a)` от самой группы). Каждый выбранный объект **статистически представляет** `(1-a)N / (bN) = (1-a)/b` объектов из исходной группы (в среднем — если сэмплирование равномерное, каждый выбранный объект «стоит» за себя и за пропущенных «похожих» соседей). Домножая градиент каждого выбранного объекта на `(1-a)/b`, мы получаем **оценку методом Хорвица–Томпсона** (стандартный статистический приём взвешивания по обратной вероятности отбора, знакомый из общей теории выборочных исследований) суммарного градиента всей «лёгкой» группы — это классический принцип **несмещённого оценивания через инвертирование вероятности отбора**.

### 4.4. Численный пример

Пусть `N=10`, модули градиентов (уже отсортированы по убыванию): `[9.0, 8.5, 6.0, 4.0, 3.5, 3.0, 2.0, 1.5, 1.0, 0.5]`.

**Параметры:** `top_rate a=0.3` -> верхние `3` объекта (`9.0, 8.5, 6.0`) берём **все**, множество `A`. `other_rate b=0.2` -> из оставшихся `7` объектов (`4.0, 3.5, 3.0, 2.0, 1.5, 1.0, 0.5`) случайно выбираем `b·N=2` объекта — пусть по жребию выпали `3.5` и `1.0`.

**Множитель:** `(1-a)/b = 0.7/0.2 = 3.5`.

**Вклад группы B в оценку суммы градиентов (с реквейтингом):**

In [ ]:
(3.5 + 1.0) · 3.5 = 4.5 · 3.5 = 15.75

**Сравним с истинной суммой всей «лёгкой» группы** (если бы мы использовали все 7 объектов честно): `4.0+3.5+3.0+2.0+1.5+1.0+0.5 = 15.5`. **Оценка `15.75` очень близка к истине `15.5`**, хотя мы физически обработали всего `2` объекта из `7` (~29% группы) — вот она, экономия вычислений почти без потери точности статистической оценки.

**А что было бы БЕЗ реквейтинга** (наивное сэмплирование, как в обычном `subsample` из Модуля 5, без компенсирующего множителя)? Оценка суммы была бы просто `3.5+1.0=4.5` — что **втрое меньше** истинных `15.5`. Такая грубая недооценка систематически искажала бы Gain для всех кандидатов на разбиение, вовлекающих эту часть данных — модель делала бы **неверные** решения о том, где резать дерево. Именно компенсирующий множитель `(1-a)/b` — то, что отличает GOSS от «выбросить часть данных наугад» и делает его статистически корректным методом, а не просто более агрессивной версией `subsample`.

**Итоговая экономия в этом примере:** обработано `3 (группа A) + 2 (группа B)` `= 5` из `10` объектов — 50% исходных данных — при этом **все** самые информативные (с наибольшим градиентом) объекты сохранены **полностью**, и лишь однородная, менее информативная часть выборки заменена статистически корректной уменьшенной оценкой.

**Ответ на чек-поинт вопрос напрямую:** GOSS ускоряет обучение **не** за счёт выбрасывания важных примеров (они как раз сохраняются все, 100%), а за счёт **уменьшения избыточности** в обработке уже хорошо предсказываемых объектов — и делает это статистически аккуратно, через реквейтинг, а не грубым случайным прореживанием всей выборки без разбора.

## 5. EFB: Exclusive Feature Bundling

### 5.1. Мотивация

После One-Hot кодирования категориальных признаков (Неделя 5) или в естественно разреженных данных (индикаторы редких событий) появляется **множество** бинарных или почти всегда нулевых столбцов, которые **редко принимают ненулевое значение одновременно** («взаимоисключающие» или «почти взаимоисключающие» признаки). Строить для каждого такого признака **отдельную** гистограмму — расточительно: большая часть корзин будет практически пустой.

### 5.2. Идея объединения через сдвиг диапазона

Если признак `A` принимает значения в диапазоне `[0, max_A]`, а признак `B` — в `[0, max_B]`, и они **никогда** не равны нулю одновременно (строго взаимоисключающие), их можно **безопасно** объединить в одну колонку `bundle`, сдвинув диапазон `B` так, чтобы он не пересекался с диапазоном `A`:

In [ ]:
offset = max_A + 1
bundle_value = A_value,               если A_value ≠ 0
bundle_value = B_value + offset,      если B_value ≠ 0 (а A_value = 0)
bundle_value = 0,                     если оба равны 0

### 5.3. Численный пример

Пусть `is_visa ∈ {0,1}` и `is_mastercard ∈ {0,1}` — два one-hot столбца **одного и того же** категориального признака `card4` (значит, они **строго** взаимоисключающие — карта не может быть одновременно `visa` и `mastercard`). `max_A = 1` (максимум `is_visa`), значит `offset = 2`.

| Транзакция | is_visa | is_mastercard | bundle |
|---|---|---|---|
| 1 | 1 | 0 | 1 |
| 2 | 0 | 1 | 0 + 2 + 1 = 3 |
| 3 | 0 | 0 | 0 (например, Amex — третья категория, вне этой пары) |

Итоговая колонка `bundle` принимает значения `{0, 1, 3}`, однозначно кодируя все три исходные ситуации **одним** столбцом вместо двух — количество признаков, для которых нужно строить отдельную гистограмму, снижается вдвое (в этом маленьком примере), а на практике — при объединении **десятков** взаимоисключающих one-hot столбцов в несколько «бандлов» — эффект гораздо заметнее.

### 5.4. Как выбираются бандлы, если признаки не идеально взаимоисключающие

Строгая взаимоисключаемость (как у one-hot столбцов одной категориальной переменной) — частный случай. В общем случае LightGBM допускает **небольшой процент конфликтов** (случаев, когда оба признака в паре одновременно ненулевые) — задача «какие признаки объединять в бандлы, чтобы суммарный конфликт был мал» формально эквивалентна задаче **раскраски графа** (признаки — вершины, вес ребра — частота одновременной ненулевости пары), которая точно решается за экспоненциальное время. LightGBM использует **жадную эвристику**: сортирует признаки по общей «степени конфликтности», затем последовательно пытается добавить каждый признак в уже существующий бандл, если это не превышает допустимый порог конфликтов, иначе — открывает новый бандл. Это не гарантирует глобально оптимальное разбиение на бандлы, но находит **достаточно хорошее** за приемлемое время — классический компромисс «жадная эвристика вместо точного NP-трудного решения», с которым вы ещё не раз встретитесь в отдельном алгоритмическом треке (Дейкстра, DSU, жадные алгоритмы — но это, как отмечено в плане курса, отдельная тема, не бустинг).

## 6. Ключевые гиперпараметры LightGBM — свод и связь с уже известными концепциями

| Параметр | Физический смысл | Связь с уже пройденным |
|---|---|---|
| `num_leaves` | Максимальное число листьев в дереве — **главный** регулятор сложности | Аналог `max_depth` из Модуля 1, но для leaf-wise роста (раздел 3.3) |
| `max_depth` | Дополнительное ограничение глубины (подстраховка) | То же понятие, что в Модуле 1, но вторичное по важности здесь |
| `min_data_in_leaf` | Минимум объектов в листе | Прямой аналог `min_samples_leaf` (Модуль 1) — защита от переобучения на мелких, статистически ненадёжных группах |
| `learning_rate` | Размер шага при аддитивном обновлении | Модуль 5, раздел 2 — та же величина `η` |
| `n_estimators` | Число деревьев (итераций бустинга) | Модуль 5, раздел 4 — контролирует итоговую сложность ансамбля, нужна ранняя остановка |
| `feature_fraction` | Доля признаков, случайно рассматриваемых на каждой итерации/дереве | Аналог `max_features` (Модуль 2) — декорреляция и регуляризация |
| `bagging_fraction` / `bagging_freq` | Доля строк для случайного сэмплирования (обычная стохастика, не GOSS) и частота её обновления | Аналог `subsample` (Модуль 5) |
| `lambda_l1`, `lambda_l2` | L1/L2-регуляризация весов листьев | Прямое продолжение `λ` из формулы Модуля 6 (L2) плюс L1-аналог (штраф на модуль веса, а не квадрат — способствует более разреженным, менее экстремальным весам листьев) |
| `min_gain_to_split` | Минимальный Gain, при котором разбиение вообще делается | Прямой аналог `γ` из формулы Gain (Модуль 6) |
| `top_rate`, `other_rate` | Параметры `a` и `b` алгоритма GOSS (раздел 4) | Специфично для LightGBM |

**Практическое следствие таблицы:** несмотря на новые названия, **почти все** гиперпараметры LightGBM — это переименованные или слегка расширенные версии концепций, которые вы уже строго вывели и понимаете из Модулей 1–6. Это не новый материал для запоминания с нуля, а применение уже усвоенной логики к конкретному API библиотеки.

## 7. Практика: код

### 7.1. `num_leaves=31` vs `num_leaves=127` при `max_depth=-1` — переобучение

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.metrics import average_precision_score

X, y = make_classification(n_samples=50000, n_features=40, n_informative=20,
                            weights=[0.95, 0.05], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

for num_leaves in [7, 31, 127, 500]:
    model = lgb.LGBMClassifier(
        num_leaves=num_leaves,
        max_depth=-1,          # без ограничения глубины - вся регуляризация через num_leaves
        n_estimators=200,
        learning_rate=0.05,
        random_state=42,
        verbose=-1,
    )
    model.fit(X_train, y_train)

    train_pr_auc = average_precision_score(y_train, model.predict_proba(X_train)[:, 1])
    test_pr_auc = average_precision_score(y_test, model.predict_proba(X_test)[:, 1])

    print(f"num_leaves={num_leaves:4d} | train PR-AUC={train_pr_auc:.4f} | "
          f"test PR-AUC={test_pr_auc:.4f} | gap={train_pr_auc-test_pr_auc:.4f}")

**Что ожидать:** при `num_leaves=7` — недообучение (train и test PR-AUC близки, но оба невысокие). При `num_leaves=31` (значение по умолчанию, не случайно) — разумный баланс. При `num_leaves=127` и особенно `500` — train PR-AUC продолжит расти, но **разрыв** (`gap`) между train и test начнёт увеличиваться — классический признак переобучения, полностью в духе того, что мы теоретически предсказали в разделе 3.3.

### 7.2. Эффект GOSS — сравнение времени обучения

In [ ]:
import time

for boosting_type in ["gbdt", "goss"]:
    start = time.perf_counter()
    model = lgb.LGBMClassifier(
        boosting_type=boosting_type,
        num_leaves=31,
        n_estimators=300,
        learning_rate=0.05,
        random_state=42,
        verbose=-1,
    )
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - start

    test_pr_auc = average_precision_score(y_test, model.predict_proba(X_test)[:, 1])
    print(f"boosting_type={boosting_type:5s} | время={elapsed:.2f}с | test PR-AUC={test_pr_auc:.4f}")

*(Примечание: в новых версиях LightGBM параметр `boosting_type='goss'` может выводиться из употребления в пользу единого движка с внутренними флагами — если такой конструктор недоступен в вашей версии библиотеки, обратитесь к актуальной документации `lightgbm` за эквивалентным способом включения GOSS.)*

### 7.3. Демонстрация EFB — влияние на память/скорость при высококардинальных one-hot признаках

In [ ]:
import numpy as np
import pandas as pd

# Симулируем сильно разреженные one-hot колонки одной категориальной переменной
n_samples = 50000
categories = np.random.choice(["visa", "mastercard", "mir", "amex"], size=n_samples,
                               p=[0.5, 0.3, 0.15, 0.05])
onehot_df = pd.get_dummies(pd.Series(categories), prefix="card")

print(onehot_df.head())
print("Доля ненулевых значений в каждой колонке (наглядно видна разреженность):")
print(onehot_df.mean())

LightGBM автоматически применит EFB к таким колонкам внутри `.fit()` — специально включать ничего не нужно, но полезно самостоятельно увидеть, **насколько** разрежены типичные one-hot признаки (обычно доля единиц — от нескольких процентов до половины, редко больше) — именно эта разреженность и делает EFB эффективным.

## 8. Частые вопросы на собеседовании

| Вопрос | На что обратить внимание в ответе |
|---|---|
| Почему при leaf-wise росте `num_leaves` важнее, чем `max_depth`? | Leaf-wise рост может создавать асимметричные деревья — глубина одной ветки не отражает общую сложность модели; именно число листьев напрямую определяет количество независимых решающих правил |
| Как GOSS ускоряет обучение, не выбрасывая важные примеры? | Все объекты с большим градиентом (наиболее информативные) сохраняются полностью; из «лёгких» объектов делается статистически корректная (реквейтинг через `(1-a)/b`) уменьшенная выборка, сохраняющая несмещённость оценки суммарного градиента |
| В чём разница между `subsample` (Модуль 5) и GOSS? | `subsample` — равномерное случайное прореживание без учёта важности объектов; GOSS — целенаправленное сохранение информативных (высокоградиентных) объектов и умное, статистически скорректированное сэмплирование остальных |
| Зачем нужен EFB, если признаки и так проходят через гистограммы? | Гистограммы всё равно требуют памяти и вычислений на каждый отдельный признак; объединение взаимоисключающих разреженных признаков в бандлы снижает число признаков, для которых нужно строить отдельные гистограммы, почти без потери информации |
| Что произойдёт, если поставить `num_leaves` очень большим при `max_depth=-1`? | Дерево сможет расти очень глубоко в отдельных ветках, находя разбиения, выгодные на всё более мелких подвыборках — риск сильного переобучения, аналогично неограниченному одиночному дереву из Модуля 1 |

## 9. Чек-поинт — попробуйте ответить без подсказок

1. Почему при leaf-wise росте `num_leaves` важнее, чем `max_depth`?
2. Как GOSS ускоряет обучение, не выбрасывая важные примеры?
3. Объясните трюк вычитания гистограмм — почему для получения гистограммы одного из двух потомков не нужен отдельный проход по данным?
4. Почему в GOSS объекты с маленьким градиентом при сэмплировании умножаются именно на `(1-a)/b`, а не на какой-то другой коэффициент?
5. При каком условии (с точки зрения бюджета листьев) leaf-wise и level-wise рост дают идентичный результат, а при каком — расходятся?

## Ответы для самопроверки

<details>
<summary>Раскрыть после того, как попробуете ответить сами</summary>

1. Leaf-wise рост выбирает для разбиения лист с максимальным Gain **независимо** от его текущей глубины — в результате дерево может стать сильно несимметричным: одна ветвь уходит очень глубоко, другая почти не растёт. Глубина отдельной ветки в таком дереве **не отражает** общую сложность всей модели, поэтому ограничение по `max_depth` плохо контролирует переобучение. `num_leaves`, наоборот, напрямую задаёт число итоговых «решающих правил» дерева — это и есть настоящая мера его сложности при leaf-wise росте.

2. GOSS **не отбрасывает** объекты с большим по модулю градиентом (наиболее информативные для текущей итерации, показывающие, где модель ошибается сильнее всего) — они сохраняются на 100%. Только для объектов с маленьким градиентом (модель их и так уже неплохо предсказывает) применяется случайное сэмплирование, но с компенсирующим множителем `(1-a)/b`, который восстанавливает несмещённую оценку суммарного вклада всей этой группы — то есть статистическая точность оценки градиентов сохраняется, а вычислений требуется меньше.

3. Гистограмма узла — это набор сумм (`G`, `H`) по корзинам признака для всех объектов, попавших в узел. Так как множества объектов левого и правого потомка **не пересекаются** и **в сумме составляют** множество объектов родителя, справедливо тождество `Гистограмма(родитель) = Гистограмма(левый) + Гистограмма(правый)`. Если гистограмма родителя уже известна, а гистограмма одного (обычно меньшего) потомка посчитана явным проходом по его данным, гистограмма второго потомка получается простым вычитанием, без дополнительного прохода по его объектам.

4. Множитель `(1-a)/b` — это оценка методом инвертирования вероятности отбора (метод Хорвица–Томпсона): из `(1-a)N` объектов «лёгкой» группы отбирается `bN`, то есть доля `b/(1-a)` от размера самой группы. Каждый отобранный объект статистически «представляет» `(1-a)/b` объектов исходной группы (обратная величина вероятности отбора) — домножение на этот коэффициент делает сумму по выборке несмещённой оценкой суммы по всей группе, а не просто заниженной суммой по неполной подвыборке.

5. Если бюджет листьев позволяет забрать **все** доступные на текущем уровне выгодные разбиения (то есть все кандидаты действительно стоит разбивать), leaf-wise и level-wise в итоге построят одно и то же дерево — просто разным порядком действий. Расхождение возникает именно тогда, когда бюджет **ограничен** и **не позволяет** взять все доступные на уровне разбиения одновременно — тогда leaf-wise жадно берёт разбиения с наибольшим Gain первыми (используя ограниченный бюджет максимально эффективно), а level-wise вынужден либо забрать весь уровень целиком (включая невыгодные разбиения), либо не заходить на этот уровень вовсе (упуская выгодные), так как не умеет выбирать частично внутри одного уровня.

</details>